In [ ]:
# setup
import io
import base64
import sys
import glob
import errno
from collections import defaultdict
import os
import gc

import numpy as np
import cupy as cp
import h5py
import scipy as sp
import pandas as pd
import itertools
import multiprocessing as mproc
import pandas as pd
import PIL as pil
import rembg
import glob
import calendar

sys.path.append(os.getcwd())

%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML, Math, Latex
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

import matplotlib.pyplot as plt
# from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import matplotlib as mpl
# from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline
#%matplotlib notebook
    
# mpl.rcParams['text.usetex'] = 'True'
mpl.rcParams['axes.grid'] = False
mpl.rcParams['xtick.top'] = True
mpl.rcParams['xtick.bottom'] = True
mpl.rcParams['ytick.left'] = True
mpl.rcParams['ytick.right'] = True
mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.major.size'] = 14
mpl.rcParams['ytick.major.size'] = 14
mpl.rcParams['xtick.minor.size'] = 7
mpl.rcParams['ytick.minor.size'] = 7
mpl.rcParams['xtick.major.width'] = 2
mpl.rcParams['ytick.major.width'] = 2
mpl.rcParams['xtick.minor.width'] = 1.2
mpl.rcParams['ytick.minor.width'] = 1.2
mpl.rcParams['axes.labelsize'] = 20
mpl.rcParams['xtick.labelsize'] = 20
mpl.rcParams['ytick.labelsize'] = 20
mpl.rcParams['legend.loc'] = 'best'
mpl.rcParams['legend.fontsize'] = 25
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.serif'] = 'Computer Modern'
mpl.rcParams['xtick.major.pad']='8'
mpl.rcParams['ytick.major.pad']='8'
mpl.rcParams['ytick.major.pad']='8'

blue = '#1f77b4'
orange = '#ff7f0e'
green = '#2ca02c'
red = '#ad494a'
violet = '#9467bd'
brown = '#8c564b'

In [1]:
from jsd import jsd
db = jsd('data/All_crop.xlsx')

In [2]:
db.is_cultivation_time('July')
# db.is_crop('Tomato')

,Variety,Crop,Hybrid or OP,Sowing and transplant starting time,Sowing and transplant ending time,Best sowing and transplant starting time,Best sowing and transplant ending time,Maturity (days),After sowing or transplant,Weight (g),Fruit Size (cm),"Fruit shape, type, color",Disease Tolerance,Weather tolerance,Transport and storage property
0,F1 Hybrid Hot Pepper Hot Star,Hot Pepper,hybrid,January,December,January,December,50,transplant,13-15,11,"long, slim, red (ripe)",,heat,
1,F1 Hybrid Hot Pepper Balijuri,Hot Pepper,hybrid,January,December,January,December,50,transplant,7-9,10-12,glossy green to red,Yes,"heat, rain",
2,F1 Hybrid Hot Pepper Bengal Hot,Hot Pepper,hybrid,January,December,January,December,50-55,transplant,5,10,"glossy green, straight",,"heat, rain",
4,F1 Hybrid Hot Pepper Mohona,Hot Pepper,hybrid,January,December,January,December,50-55,transplant,10,13,green,,,
5,F1 Hybrid Hot Pepper Mohona-2,Hot Pepper,hybrid,January,December,December,February,50-55,transplant,8-10,10-12,green,Yes,,
6,F1 Hybrid Hot Pepper Siam Hot,Hot Pepper,hybrid,January,December,April,August,55-65,transplant,8-10,10-13,glossy green,Yes,"heat, rain",
7,F1 Hybrid Hot Pepper Thai Hot,Hot Pepper,hybrid,January,December,April,June,50,transplant,9-10,13,dark green,Yes,,
9,F1 Hybrid Tomato Beautiful,Tomato,hybrid,January,December,January,December,50-55,transplant,150,,oval,"BW, Nematode, TYLCV",heat tolerant,good
10,F1 Hybrid Tomato Beautiful-2,Tomato,hybrid,January,December,January,December,65,transplant,135,7.2,oval,"BW, FW, TYLCV, LB",,good
15,F1 Hybrid Tomato Profit Early,Tomato,hybrid,May,August,May,August,50,transplant,135,,"oval, firm","BW, FW, TYLCV, TMV",heat tolerant,good


In [3]:
db.dataframe

,Variety,Crop,Hybrid or OP,Sowing and transplant starting time,Sowing and transplant ending time,Best sowing and transplant starting time,Best sowing and transplant ending time,Maturity (days),After sowing or transplant,Weight (g),Fruit Size (cm),"Fruit shape, type, color",Disease Tolerance,Weather tolerance,Transport and storage property
0,F1 Hybrid Hot Pepper Hot Star,Hot Pepper,hybrid,January,December,January,December,50,transplant,13-15,11,"long, slim, red (ripe)",,heat,
1,F1 Hybrid Hot Pepper Balijuri,Hot Pepper,hybrid,January,December,January,December,50,transplant,7-9,10-12,glossy green to red,Yes,"heat, rain",
2,F1 Hybrid Hot Pepper Bengal Hot,Hot Pepper,hybrid,January,December,January,December,50-55,transplant,5,10,"glossy green, straight",,"heat, rain",
3,F1 Hybrid Hot Pepper Bindhu,Hot Pepper,hybrid,December,February,December,February,50,transplant,3-4,7-8,glossy green,,,
4,F1 Hybrid Hot Pepper Mohona,Hot Pepper,hybrid,January,December,January,December,50-55,transplant,10,13,green,,,
5,F1 Hybrid Hot Pepper Mohona-2,Hot Pepper,hybrid,January,December,December,February,50-55,transplant,8-10,10-12,green,Yes,,
6,F1 Hybrid Hot Pepper Siam Hot,Hot Pepper,hybrid,January,December,April,August,55-65,transplant,8-10,10-13,glossy green,Yes,"heat, rain",
7,F1 Hybrid Hot Pepper Thai Hot,Hot Pepper,hybrid,January,December,April,June,50,transplant,9-10,13,dark green,Yes,,
8,F1 Hybrid Capsicum Green Bell,Capsicum,hybrid,December,February,December,February,65,transplant,200,10,"bell shape, green",,heat,
9,F1 Hybrid Tomato Beautiful,Tomato,hybrid,January,December,January,December,50-55,transplant,150,,oval,"BW, Nematode, TYLCV",heat tolerant,good
